In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision.models import vit_l_16, ViT_L_16_Weights
import torchvision.transforms as transforms
from sklearn.model_selection import GroupShuffleSplit
from torch.optim.lr_scheduler import CosineAnnealingLR

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, roc_curve
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
base_path = Path("/kaggle/input/faceswap-faces/")
batch_size = 4

In [ ]:
df = pd.read_csv(base_path / "processed_metadata.csv")
df.head()
    

In [ ]:
video_labels = df.groupby('video_id')['label'].first().reset_index()
video_labels


In [ ]:
video_labels.value_counts()


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=69)
train_idx, temp_idx = next(gss.split(video_labels, video_labels['label'], groups=video_labels['video_id']))

train_videos = video_labels.iloc[train_idx]
temp_videos = video_labels.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=69)
val_idx, test_idx = next(gss2.split(temp_videos, temp_videos['label'], groups=temp_videos['video_id']))

val_videos = temp_videos.iloc[val_idx]
test_videos = temp_videos.iloc[test_idx]

train_df = df[df['video_id'].isin(train_videos['video_id'])]
val_df = df[df['video_id'].isin(val_videos['video_id'])]
test_df = df[df['video_id'].isin(test_videos['video_id'])]

train_df


In [ ]:
print("Train/Val/Test split summary:")
print(train_df['label'].value_counts())
print(val_df['label'].value_counts())
print(test_df['label'].value_counts())


In [ ]:
train_vids = set(train_df['video_id'].unique())
val_vids = set(val_df['video_id'].unique())
test_vids = set(test_df['video_id'].unique())

print("Train∩Val:", len(train_vids & val_vids))
print("Train∩Test:", len(train_vids & test_vids))
print("Val∩Test:", len(val_vids & test_vids))

train_files = set(train_df['filepath'])
val_files   = set(val_df['filepath'])
print("Common files train/val:", len(train_files & val_files))


In [ ]:
def custom_collate_fn(batch):
    imgs, labels, vids = zip(*batch)
    imgs = torch.stack(imgs)
    labels = torch.stack(labels)
    return imgs, labels, vids


In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, data, seq_length=50):
        self.df = data
        self.seq_length = seq_length
        self.videos = data['video_id'].unique()

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        vid = self.videos[idx]
        frames = self.df[self.df['video_id'] == vid]
        if len(frames) == 0:
            print(f"[WARN] Label {frames['label'][0]} Video {vid} has 0 frames in DataFrame")

        imgs = [Image.open(f"/kaggle/input/faceswap-faces/{fp}").convert('RGB') for fp in frames['filepath']]
        imgs = [self.transform(i) for i in imgs]
        imgs = torch.stack(imgs)
        imgs = self._add_padding(imgs)
        label = 1 if frames['label'].iloc[0] == 'FAKE' else 0
        vid = str(vid)
        return imgs, torch.tensor(label), vid


    def transform(self, img):
        if img.size != (224, 224):
            img = img.resize((224, 224))
        img_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomResizedCrop(224,scale=(0.8, 1.0)),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
        ])
        torch_tensor = img_transform(img)
        return torch_tensor

    def _add_padding(self, images):
        if images.size(0) > self.seq_length:
            images = images[:self.seq_length]
        padding_needed = self.seq_length - images.size(0)
        if padding_needed > 0:
            pad_tensor = torch.zeros(padding_needed, 3, 224, 224)
            images = torch.cat((images, pad_tensor), dim=0)
        return images


In [ ]:
train_dataset = DeepfakeDataset(train_df, seq_length=64)
val_dataset = DeepfakeDataset(val_df, seq_length=64)
test_dataset = DeepfakeDataset(test_df, seq_length=64)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle=True, num_workers=4, pin_memory=True)


In [ ]:
NUM_CLASSES = 2                  # real / fake (change if needed)
T_FRAMES = 64
IMG_SIZE = 112                   # resize to 112x112 (adjust)
BATCH_SIZE = 8                    # adjust to fit GPU memory
LR = 3e-4
WEIGHT_DECAY = 1e-5
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
class SimpleFrameCNN(nn.Module):
    """Small 2D CNN that maps a single frame to a feature vector."""
    def __init__(self, emb_dim=512):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.MaxPool2d(2),  # 56x56
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(2),  # 28x28
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(256),
            nn.AdaptiveAvgPool2d(1),      # -> (256,1,1)
        )
        self.fc = nn.Linear(256, emb_dim)

    def forward(self, x):
        # x: (B, 3, H, W)
        x = self.conv(x)
        x = x.view(x.size(0), -1)   # (B, 256)
        x = self.fc(x)              # (B, emb_dim)
        return x

In [ ]:
class MeanPoolSequenceModel(nn.Module):
    def __init__(self, emb_dim=512, num_classes=NUM_CLASSES):
        super().__init__()
        self.frame_cnn = SimpleFrameCNN(emb_dim=emb_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        # process all frames in one pass by merging B and T
        x = x.view(B * T, C, H, W)          # (B*T, C, H, W)
        feats = self.frame_cnn(x)           # (B*T, emb_dim)
        feats = feats.view(B, T, -1)        # (B, T, emb_dim)
        pooled = feats.mean(dim=1)          # mean pooling across time -> (B, emb_dim)
        logits = self.classifier(pooled)    # (B, num_classes)
        return logits


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

model = MeanPoolSequenceModel()
if device == "cuda":
    model = model.to(device)

In [ ]:
num_epochs = 50
criterion = nn.BCEWithLogitsLoss(pos_weight=None, reduction='mean')  # More stable for binary tasks
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr = 1e-4, steps_per_epoch=len(train_loader), epochs=num_epochs)


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, verbose=False):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        torch.save(model.state_dict(), 'checkpoint.pt')
        self.val_loss_min = val_loss

In [ ]:
scaler = torch.amp.GradScaler('cuda')
early_stopping = EarlyStopping(patience=7, verbose=True)

train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    print("\n" + "="*50)
    print(f"Running Epoch: {epoch+1}/{num_epochs}")
    print("="*50)
    
    # ------------------
    # 1. Training Phase (Sequence-level)
    # ------------------
    model.train()
    train_loss = 0
    
    # imgs shape: (B, T, C, H, W), labels shape: (B), _ (vids)
    for imgs, labels, _ in tqdm(train_loader, desc="Training", file=sys.stdout):
        
        # Move video sequence batch to device
        imgs = imgs.to(device)
        labels = labels * 0.9 + 0.05
        
        # Move and format labels: (B) -> (B, 1) float for BCEWithLogitsLoss
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        
        # Autocast enables mixed precision for the forward pass
        with torch.amp.autocast(device):
            # Forward pass: model expects (B, T, C, H, W) and outputs (B, 1)
            outputs = model(imgs) 
            loss = criterion(outputs, labels)

        # Scale loss, backpropagate, and update
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    
    # Step the learning rate scheduler after the epoch
    
    # ------------------
    # 2. Validation Phase (Sequence-level)
    # ------------------
    model.eval()
    per_video_rows = []
    val_loss = 0
    
    with torch.no_grad(), torch.amp.autocast(device):
        for imgs, labels, vids in val_loader:
            B, _, _, _, _ = imgs.shape
            
            # Pass the entire sequence batch (B, T, C, H, W) to the model
            imgs = imgs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            if torch.rand(1).item() < 0.3:
                imgs = imgs[:, torch.randperm(imgs.size(1)), ...]
            
            # Forward pass, outputs are logits (B, 1)
            outputs = model(imgs) 
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            
            # Apply sigmoid to get video prediction probabilities and flatten (B,)
            video_preds = torch.sigmoid(outputs).cpu().numpy().ravel()
            
            for i in range(B):
                per_video_rows.append({
                    'video_id': vids[i],
                    # Direct score from the model's output
                    'pred': float(video_preds[i]), 
                    'label': int(labels[i].item())
                })
    avg_val_loss = val_loss / len(val_loader)
    
    # --- Evaluation ---
    video_df_pred = pd.DataFrame(per_video_rows)
    # Only keep one prediction per video, although the current loader only provides one.
    video_df_pred = video_df_pred.drop_duplicates(subset='video_id') 
    
    # Ensure there are enough samples and both classes for meaningful metrics
    if len(video_df_pred) > 1 and len(video_df_pred['label'].unique()) > 1:
        auc = roc_auc_score(video_df_pred['label'], video_df_pred['pred'])
    else:
        auc = 0.0
        print("Warning: Insufficient data for AUC calculation.")

    acc = accuracy_score(video_df_pred['label'], (video_df_pred['pred'] > 0.5).astype(int))

    early_stopping(avg_val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

    train_losses.append(avg_train_loss) 
    val_losses.append(avg_val_loss)
    val_accuracies.append(acc)

    # ------------------
    # 3. Epoch Summary
    # ------------------
    print("\n" + "*"*60)
    print(f"Epoch {epoch+1} Summary:")
    print(f"| Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {auc:.4f} | Val Acc: {acc:.4f} |")
    # print("\nPrediction Score Distribution:")
    # print(video_df_pred['pred'].describe().to_string())
    print("\n" + "*"*60)


In [ ]:
def test_model(model):
    model.eval()
    per_video_rows = []
    test_loss = 0
    
    with torch.no_grad():
        for imgs, labels, vids in test_loader:
            B, _, _, _, _ = imgs.shape
            
            imgs = imgs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            
            video_preds = torch.sigmoid(outputs).cpu().numpy().ravel()
            
            for i in range(B):
                per_video_rows.append({
                    'video_id': vids[i],
                    'pred': float(video_preds[i]),
                    'label': int(labels[i].item())
                })
    
    avg_test_loss = test_loss / len(test_loader)
    
    # --- Evaluation ---
    video_df_pred = pd.DataFrame(per_video_rows).drop_duplicates(subset='video_id')
    
    if len(video_df_pred) > 1 and len(video_df_pred['label'].unique()) > 1:
        auc = roc_auc_score(video_df_pred['label'], video_df_pred['pred'])
    else:
        auc = 0.0
        print("Warning: Insufficient data for AUC calculation.")

    true_labels = video_df_pred['label']
    binary_predictions = (video_df_pred['pred'] > 0.5).astype(int)
    acc = accuracy_score(true_labels, binary_predictions)
    report = classification_report(true_labels, binary_predictions, target_names=['Real', 'Fake'])

    # --- Print Summary ---
    print("\n" + "*"*60)
    print("Test Summary:")
    print(f"| Test Loss: {avg_test_loss:.4f} | Test AUC: {auc:.4f} | Test Acc: {acc:.4f} |")
    print("\nClassification Report:")
    print(report)
    print("*"*60)

    # --- Plot Confusion Matrix ---
    cm = confusion_matrix(true_labels, binary_predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.show()

    # --- Plot ROC Curve ---
    fpr, tpr, _ = roc_curve(true_labels, video_df_pred['pred'])
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.show()

# --- Run the test function ---
test_model(model)


In [ ]:
torch.save(model.state_dict(), "model_state.pth")


In [ ]:
# Create a new instance of the model
model2 = DetectionModel()

# Load the saved state dictionary
checkpoint_dict = torch.load("checkpoint.pt")

# Load the state dict into the model IN-PLACE
model2.load_state_dict(checkpoint_dict)

# Now, move the model to the correct device
if device == "cuda":
    model2 = model2.to(device)

# Test the model
test_model(model2)


In [ ]:
import matplotlib.pyplot as plt

num_epochs_run = len(train_losses) 
epochs = range(1, num_epochs_run + 1) # X-axis data

# Plot 1: Loss Curves
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses, label='Training Loss', marker='.')
plt.plot(epochs, val_losses, label='Validation Loss', marker='.')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Plot 2: Accuracy Curve
plt.figure(figsize=(10, 5))
plt.plot(epochs, val_accuracies, label='Validation Accuracy', color='green', marker='.')
plt.title('Validation Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()
